## Retrieve activity data from Strava for storage and analysis

This notebook demonstrates how to retrieve activity data from Strava for storage and analysis. It uses the stravalib library to authenticate with Strava and retrieve activity data.


## 1. Connecting to the Supabase Database

In [5]:
import os
from dotenv import load_dotenv
import psycopg2
import pandas as pd
from stravalib import Client

# Load the keys out of your hidden .env file
load_dotenv()

# Safely extract the secret strings
client_id = os.getenv("STRAVA_CLIENT_ID")
client_secret = os.getenv("STRAVA_CLIENT_SECRET")

# Connect to Supabase using the hidden environmental variables
conn = psycopg2.connect(
    host=os.getenv("SUPABASE_HOST"),
    port="6543",
    user=os.getenv("SUPABASE_USER"),
    password=os.getenv("SUPABASE_PASSWORD"),
    database="postgres"
)
cursor = conn.cursor()

## 2. Authenticating with the Strava API

In [6]:
from stravalib import Client

# Set up Strava client
client = Client()

# Authenticate with Strava
client_id = client_id
client_secret = client_secret
redirect_uri = "http://localhost/"  # Must match the callback domain in your Strava app

# Generate the authorization URL
auth_url = client.authorization_url(
    client_id=int(client_id),
    redirect_uri=redirect_uri,
    scope="activity:read_all"  # Request read access to all activities
)

print(f"Go to this URL to authorize: {auth_url}")

# Get the authorization code from the URL
code = input("Enter the authorization code from the URL: ")

# Exchange the code for an access token
token = client.exchange_code_for_token(
    client_id=int(client_id),
    client_secret=client_secret,
    code=code
)

# Save the access token
TOKEN = token["access_token"]


Go to this URL to authorize: https://www.strava.com/oauth/authorize?client_id=267964&redirect_uri=http%3A%2F%2Flocalhost%2F&approval_prompt=auto&scope=activity%3Aread_all&response_type=code


## 3. Collecting Athlete Data

In [7]:
import requests

# Collecting athlete data from Strava API
athlete_url = "https://www.strava.com/api/v3/athlete?access_token=" + TOKEN
response_athlete = requests.get(athlete_url)
athlete_data = response_athlete.json()

# Extract athlete details
athlete_name = athlete_data["firstname"] + " " + athlete_data["lastname"]
gender = athlete_data["sex"]

## Strava API v3 does not expose age or birthday â€” enter manually
#age_input = input("Enter your age (or press Enter to skip): ").strip()
#age = int(age_input) if age_input.isdigit() else None

# Verify if the athlete is already in the database
cursor.execute("SELECT id FROM athletes WHERE athlete_name = %s", (athlete_name,))
athlete = cursor.fetchone()

if athlete is None:
    # Insert new athlete
    cursor.execute(
        "INSERT INTO athletes (athlete_name, gender, age) VALUES (%s, %s, %s) RETURNING id",
        (athlete_name, gender, age)
    )
    conn.commit()
    athlete_id = cursor.fetchone()[0]  # Get the new athlete ID
else:
    athlete_id = athlete[0]  # Use the existing athlete ID

## 4. Collecting activity data

In [8]:
import time

activities_url = "https://www.strava.com/api/v3/athlete/activities?access_token=" + TOKEN
activities = []
page = 1
per_page = 150  # Number of activities per page

while True:
    # Fetch activities with pagination
    paginated_url = f"{activities_url}&page={page}&per_page={per_page}"
    response_activities = requests.get(paginated_url)
    data = response_activities.json()

    if not data:
        break  # Stop if no more activities are returned

    activities.extend(data)  # Add activities to the list
    page += 1  # Move to the next page

    time.sleep(1)  # Avoid hitting the API rate limit

## 5. Collecting splits data

In [9]:
import time

splits_data = []
failed_ids = []

print(f"Fetching splits for {len(activities)} activities...")

for i, activity in enumerate(activities):
    strava_id = activity['id']
    detail_url = f"https://www.strava.com/api/v3/activities/{strava_id}?access_token={TOKEN}"
    response = requests.get(detail_url)

    if response.status_code == 429:
        print(f"Rate limit hit at activity {i+1}. Waiting 60s...")
        time.sleep(60)
        response = requests.get(detail_url)

    if response.status_code == 200:
        for split in response.json().get('splits_metric', []):
            splits_data.append({
                'strava_activity_id': strava_id,
                'split_number': split['split'],
                'distance_km': round(split['distance'] / 1000, 4),
                'elapsed_time_s': split['elapsed_time'],
                'moving_time_s': split['moving_time'],
                'elevation_diff_m': split.get('elevation_difference', 0.0),
                'avg_speed_kmh': round(split['average_speed'] * 3.6, 3),
                'avg_heartrate': split.get('average_heartrate'),
                'pace_zone': split.get('pace_zone'),
            })
    else:
        failed_ids.append(strava_id)

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(activities)} processed...")

    time.sleep(0.5)  # Stay within Strava rate limit (200 req/15 min)

splits_df = pd.DataFrame(splits_data) if splits_data else pd.DataFrame()
print(f"Done. {len(splits_df)} splits collected across {len(activities) - len(failed_ids)} activities.")
if failed_ids:
    print(f"Failed IDs: {failed_ids}")


Fetching splits for 369 activities...
  25/369 processed...
  50/369 processed...
  75/369 processed...
Rate limit hit at activity 96. Waiting 60s...
Rate limit hit at activity 97. Waiting 60s...
Rate limit hit at activity 98. Waiting 60s...
Rate limit hit at activity 99. Waiting 60s...
Rate limit hit at activity 100. Waiting 60s...
  100/369 processed...
  125/369 processed...
  150/369 processed...
  175/369 processed...
Rate limit hit at activity 200. Waiting 60s...
  200/369 processed...
Rate limit hit at activity 201. Waiting 60s...
Rate limit hit at activity 202. Waiting 60s...
Rate limit hit at activity 203. Waiting 60s...
Rate limit hit at activity 204. Waiting 60s...
Rate limit hit at activity 205. Waiting 60s...
Rate limit hit at activity 206. Waiting 60s...
Rate limit hit at activity 207. Waiting 60s...
Rate limit hit at activity 208. Waiting 60s...
Rate limit hit at activity 209. Waiting 60s...
Rate limit hit at activity 210. Waiting 60s...
Rate limit hit at activity 211. W

## 6. Transforming the Data

In [10]:
import pandas as pd
from datetime import datetime
import pytz

# Convert data to a DataFrame
df = pd.DataFrame(activities)

# Handle optional columns that may not be present for all activities
for col in ['average_heartrate', 'suffer_score', 'average_watts', 'kilojoules', 'start_latlng', 'end_latlng']:
    if col not in df.columns:
        df[col] = None

# Select relevant columns
df = df[['id', 'name', 'start_date', 'type', 'distance', 'moving_time',
         'total_elevation_gain', 'max_speed', 'average_speed',
         'average_heartrate', 'suffer_score', 'average_watts', 'kilojoules',
         'start_latlng', 'end_latlng']]

# Convert units
df["distance"] = df["distance"] / 1000  # Convert meters to kilometers
df["moving_time"] = df["moving_time"] / 60  # Convert seconds to minutes
df["max_speed"] = df["max_speed"] * 3.6  # Convert m/s to km/h
df["average_speed"] = df["average_speed"] * 3.6  # Convert m/s to km/h

# Extract lat/lng floats (None when GPS is unavailable, e.g. indoor/manual activities)
df["start_lat"] = df["start_latlng"].apply(lambda x: x[0] if x else None)
df["start_lng"] = df["start_latlng"].apply(lambda x: x[1] if x else None)
df["end_lat"]   = df["end_latlng"].apply(lambda x: x[0] if x else None)
df["end_lng"]   = df["end_latlng"].apply(lambda x: x[1] if x else None)

# Adjust time zone and format for MySQL
from_zone = pytz.utc
to_zone = pytz.timezone("Europe/Rome")

def convert_to_mysql_datetime(start_date):
    start_date = start_date.rstrip('Z')
    dt = datetime.strptime(start_date, '%Y-%m-%dT%H:%M:%S')
    dt = from_zone.localize(dt).astimezone(to_zone)
    dt = dt.replace(tzinfo=None)  # Remove time zone info
    return dt

df["start_date"] = df["start_date"].apply(convert_to_mysql_datetime)

## 7. Loading data back into Supabase

In [11]:
def safe(val):
    if not pd.notna(val):
        return None
    if hasattr(val, 'item'):
        return val.item()
    return val

for _, row in df.iterrows():
    # Check if the activity already exists
    cursor.execute("SELECT id from activities WHERE activity_id = %s", (row["id"],))
    existing_activity = cursor.fetchone()

    if existing_activity is None:
        cursor.execute("""
            INSERT INTO activities (athlete_id, activity_name, start_date, activity_type, distance, duration, elevation,
                                    max_speed, avg_speed, avg_heartrate, suffer_score, avg_watts, kilojoules,
                                    start_lat, start_lng, end_lat, end_lng, activity_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (athlete_id, row["name"], row["start_date"], row["type"], row["distance"], row["moving_time"],
              row["total_elevation_gain"], row["max_speed"], row["average_speed"],
              safe(row["average_heartrate"]), safe(row["suffer_score"]),
              safe(row["average_watts"]), safe(row["kilojoules"]),
              safe(row["start_lat"]), safe(row["start_lng"]),
              safe(row["end_lat"]), safe(row["end_lng"]), row["id"]))
    else:
        cursor.execute("""
            UPDATE activities
            SET athlete_id = %s, activity_name = %s, start_date = %s, activity_type = %s, distance = %s,
                duration = %s, elevation = %s, max_speed = %s, avg_speed = %s,
                avg_heartrate = %s, suffer_score = %s, avg_watts = %s, kilojoules = %s,
                start_lat = %s, start_lng = %s, end_lat = %s, end_lng = %s
            WHERE activity_id = %s
        """, (athlete_id, row["name"], row["start_date"], row["type"], row["distance"], row["moving_time"],
              row["total_elevation_gain"], row["max_speed"], row["average_speed"],
              safe(row["average_heartrate"]), safe(row["suffer_score"]),
              safe(row["average_watts"]), safe(row["kilojoules"]),
              safe(row["start_lat"]), safe(row["start_lng"]),
              safe(row["end_lat"]), safe(row["end_lng"]), row["id"]))

conn.commit()
cursor.close()
conn.close()
print("Athletes and activities inserted successfully!")

Athletes and activities inserted successfully!


## 7b. Loading splits into Supabase

In [12]:
conn_splits = psycopg2.connect(
    host=os.getenv("SUPABASE_HOST"),
    port="6543",
    user=os.getenv("SUPABASE_USER"),
    password=os.getenv("SUPABASE_PASSWORD"),
    database="postgres"
)
cursor_splits = conn_splits.cursor()

inserted = 0
skipped = 0

for _, row in splits_df.iterrows():
    # Look up the database primary key for this activity using the Strava activity ID
    cursor_splits.execute("SELECT id FROM activities WHERE activity_id = %s", (int(row['strava_activity_id']),))
    result = cursor_splits.fetchone()
    if result is None:
        continue  # Activity not yet in DB; skip
    db_activity_id = result[0]

    # Skip if this split already exists (idempotent)
    cursor_splits.execute(
        "SELECT 1 FROM splits WHERE activity_id = %s AND split_number = %s",
        (db_activity_id, int(row['split_number']))
    )
    if cursor_splits.fetchone():
        skipped += 1
        continue

    cursor_splits.execute("""
        INSERT INTO splits (activity_id, split_number, distance_km, elapsed_time_s, moving_time_s,
                            elevation_diff_m, avg_speed_kmh, avg_heartrate, pace_zone)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        db_activity_id,
        int(row['split_number']),
        float(row['distance_km']),
        int(row['elapsed_time_s']),
        int(row['moving_time_s']),
        safe(row['elevation_diff_m']),
        float(row['avg_speed_kmh']),
        safe(row['avg_heartrate']),
        safe(row['pace_zone']),
    ))
    inserted += 1

conn_splits.commit()
cursor_splits.close()
conn_splits.close()
print(f"Splits loaded: {inserted} inserted, {skipped} already existed.")


Splits loaded: 68 inserted, 3237 already existed.


In [ ]:
## Check all variables saved for a given Strava activity

import json
print(json.dumps(activities[0], indent=2))
print(list(activities[0].keys()))

## 8. Loading GymNotes data into Supabase
Requires export as .csv from GymNotes app. The CSV file should have the following columns: Date, Exercise, Category, Weight, Weight Unit, Reps, Distance, Distance Unit, Time.

In [ ]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

df_fn = pd.read_csv(r"C:\Users\persif\PycharmProjects\SportAnalytics\FitNotes_Export.csv")

conn_fn = psycopg2.connect(
    host=os.getenv("SUPABASE_HOST"),
    port="6543",
    user=os.getenv("SUPABASE_USER"),
    password=os.getenv("SUPABASE_PASSWORD"),
    database="postgres"
)
cursor_fn = conn_fn.cursor()

def safe(val):
    if pd.isna(val):
        return None
    if hasattr(val, 'item'):
        return val.item()
    return val

inserted = 0
skipped = 0

for _, row in df_fn.iterrows():
    cursor_fn.execute("""
        SELECT 1 FROM fitnotes_workouts
        WHERE date = %s AND exercise = %s
          AND (weight IS NOT DISTINCT FROM %s)
          AND (reps IS NOT DISTINCT FROM %s)
          AND (distance IS NOT DISTINCT FROM %s)
    """, (
        row["Date"],
        row["Exercise"],
        safe(row["Weight"]),
        safe(row["Reps"]),
        safe(row["Distance"]),
    ))
    if cursor_fn.fetchone():
        skipped += 1
        continue

    cursor_fn.execute("""
        INSERT INTO fitnotes_workouts
            (date, exercise, category, weight, weight_unit, reps, distance, distance_unit, time_str)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row["Date"],
        row["Exercise"],
        safe(row["Category"]),
        safe(row["Weight"]),
        safe(row["Weight Unit"]),
        safe(row["Reps"]),
        safe(row["Distance"]),
        safe(row["Distance Unit"]),
        safe(row["Time"]),
    ))
    inserted += 1

conn_fn.commit()
cursor_fn.close()
conn_fn.close()
print(f"FitNotes: {inserted} rows inserted, {skipped} already existed.")